In [2]:
import os
import uuid
import numpy as np
import chromadb
from pathlib import Path
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

from langchain_core.documents import Document
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader, CSVLoader
from sentence_transformers import SentenceTransformer

### **Document Structure**

In [24]:
doc = Document(
    page_content="This is the main content which will be used to create RAG pipeline",
    metadata={
        "source": "Example.txt",
        "pages": 1,
        "author": "Kunal Sahni",
        "date_created": "2026-06-09"
    }
)

In [4]:
os.makedirs("../data/text_files", exist_ok=True)

In [5]:
sample_texts = {
    "../data/text_files/python_intro.txt": """Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, "plain English" naming, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
    Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual release cycle and five-year support policy. Python 3.15 is currently in the alpha development phase, and the stable release is expected to launch in October 2026. Earlier versions in the 3.x series have reached end-of-life and no longer receive security updates.
    Python has gained extensive use in the machine learning community. It is widely taught as an introductory programming language. Since 2003, Python has consistently ranked among the top ten most popular programming languages in the TIOBE Programming Community Index, which ranks programming languages based on searches across 24 platforms""",
    "../data/text_files/machine_learning.txt": """Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from pre-trained data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
    Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning. From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimisation under this framework."""
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created")

Sample text files created


TextLoader

In [8]:
loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
loader.load()

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, "plain English" naming, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.\n    Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project\'s annual release cycle and five-year support policy. Python 3

#### DirectoryLoader

With TextLoader

In [14]:
text_dir_loader = DirectoryLoader(
    "../data/text_files/",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

text_dir_loader.load()

100%|██████████| 2/2 [00:00<00:00, 621.47it/s]


[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from pre-trained data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.\n    Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsupervised learning. From a theoretical viewpoint, probably approximately correct learning provides a mathematical and statistical framework for describing machine learning. Most traditional machine learning and deep learning algorithms can be described as empirical risk minimi

With PyMuPDFLoader

In [ ]:
pdf_dir_loader = DirectoryLoader(
    "../data/pdf/",
    glob="**/*.pdf",
    loader_cls=PyMuPDF4LLMLoader,
    show_progress=True
)

pdf_dir_loader.load()










100%|██████████| 5/5 [00:00<00:00, 19.02it/s]


[Document(metadata={'producer': 'Pdftools SDK', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\DishaOS 5-16.pdf', 'file_path': '..\\data\\pdf\\DishaOS 5-16.pdf', 'total_pages': 19, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-15T15:25:55+00:00', 'trapped': '', 'modDate': 'D:20260415152555Z', 'creationDate': '', 'page': 0}, page_content='DISHA SAHNI                                                                                                                               00225502024 \n13 \nPractical 5 \nAim:  To understand vi basics, Three modes of vi Editor, how to \nwrite, save, execute a shell script in vi editor. \nOpening vi editor: \n \nWriting in a file: \n \nDisplaying the content of the file:'),
 Document(metadata={'producer': 'Pdftools SDK', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\DishaOS 5-16.pdf', 'file_path': '..\\data\\pdf\\DishaOS 5-16.pdf', 'total_pages': 19, 'format': 'PDF 1.7', '

With CSVLoader

In [30]:
xl_dir_loader = DirectoryLoader(
    "../data/csv/",
    glob="**/*.csv",
    loader_cls=CSVLoader,
    show_progress=True
)

xl_dir_loader.load()




100%|██████████| 4/4 [00:00<00:00, 511.39it/s]


[Document(metadata={'source': '..\\data\\csv\\10001-2023.1.18.csv', 'row': 0}, page_content='ï»¿product_id: 8198129\nproduct_name: Water Pump Melody\namount_purchased: 2\nprice_per_unit: 151\ntotal_price: 302\n: '),
 Document(metadata={'source': '..\\data\\csv\\10001-2023.1.18.csv', 'row': 1}, page_content='ï»¿product_id: 4772822\nproduct_name: Microscope Landless Land\namount_purchased: 1\nprice_per_unit: 37\ntotal_price: 37\n: '),
 Document(metadata={'source': '..\\data\\csv\\10001-2023.1.18.csv', 'row': 2}, page_content='ï»¿product_id: \nproduct_name: \namount_purchased: \nprice_per_unit: \ntotal_price: \n: '),
 Document(metadata={'source': '..\\data\\csv\\10001-2023.1.18.csv', 'row': 3}, page_content='ï»¿product_id: \nproduct_name: \namount_purchased: \nprice_per_unit: \ntotal_price: \n: '),
 Document(metadata={'source': '..\\data\\csv\\10001-2023.1.18.csv', 'row': 4}, page_content='ï»¿product_id: \nproduct_name: \namount_purchased: \nprice_per_unit: \ntotal_price: \n: '),
 Documen

### **Load PDFs**

In [12]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")
        try:
            loader = PyMuPDF4LLMLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

In [10]:
all_pdf_documents = process_all_pdfs("../../data/pdf/")

Found 5 PDF files to process
Processing DishaOS 5-16.pdf
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Loaded 19 pages
Processing SPO Owner addition - Test results.pdf
Loaded 9 pages
Processing STdisha5-6.pdf
Loaded 13 pages
Processing Udemy python certificate.pdf
Loaded 1 pages
Processing Woxen pvt ltd Disha offer letter.pdf
Loaded 2 pages
Total documents loaded: 44


In [ ]:
all_pdf_documents

### **Split into chunks**

In [13]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [15]:
chunks = split_documents(all_pdf_documents)

Split 44 documents into 47 chunks

Example chunk:
Content: # **Practical 5**

**Aim: To understand vi basics, Three modes of vi Editor, how to**

**write, save, execute a shell script in vi editor.**


Opening vi editor:


Writing in a file:


Displaying the ...
Metadata: {'producer': 'Pdftools SDK', 'creator': '', 'creationdate': '', 'source': '..\\..\\data\\pdf\\DishaOS 5-16.pdf', 'file_path': '..\\..\\data\\pdf\\DishaOS 5-16.pdf', 'total_pages': 19, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-04-15T15:25:55+00:00', 'trapped': '', 'modDate': 'D:20260415152555Z', 'creationDate': '', 'page': 0, 'source_file': 'DishaOS 5-16.pdf', 'file_type': 'pdf'}


### **Embeddings and VectorDB**

#### **Embeddings**

In [5]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) ->np.ndarray:
        """
        Generates embeddings for a list of texts

        Args:
            texts: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings wit shape {embeddings.shape}")

        return embeddings

Initialize EmbeddingManager

In [6]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2318.41it/s]


Model loaded successfully. Embedding dimension: 384


#### **VectorStore**

In [25]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initializes the vector store

        Args:
            collection_name: Name of ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initializes ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Exisiting documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [26]:
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Exisiting documents in collection: 0
